Question: what can the product team change?

Data: first visits of new users who arrived Nov 1st to Dec 24th and first visit buyers included.

In [29]:
import math
from statistics import NormalDist

import duckdb

from ga4case.path import SESSIONS, USERS

In [ ]:
con = duckdb.connect()
con.sql(
    f"create or replace view sessions as select * from read_parquet('{SESSIONS.as_posix()}')"
)
con.sql(
    f"create or replace view users as select * from read_parquet('{USERS.as_posix()}')"
);

In [31]:
# New users from Nov 1st to Dec 24th and purchased within 7 days including the first visit
con.sql("""
    create or replace view new_users as
    select *,
        (purchased_in_first_visit = 1
            or coalesce(later_purchase_start <= first_visit_end + interval 7 day, false))::int as bought_within_7_days
    from users
    where first_session_number = 1
        and first_visit_start < timestamp '2020-12-25'
""")
con.sql("""
    select count(*) as new_users,
        count(distinct first_visit_start::date) as days
    from new_users
""").df()

,new_users,days
0,154160,54


In [32]:
# New users' landing pages, arrival visitors per day and the event depth
con.sql("""
    select landing_page_title,
        count(*) as new_users,
        round(count(*) / 54.0) as per_day,
        round(100 * avg((distinct_pages <= 1)::int), 1) as pct_one_page_or_less,
        round(100 * avg(viewed_product), 1) as pct_viewed_product,
        round(100 * avg(bought_within_7_days), 2) as pct_bought_within_7_days
    from new_users
    group by 1
    order by new_users desc
    limit 6
""").df()

,landing_page_title,new_users,per_day,pct_one_page_or_less,pct_viewed_product,pct_bought_within_7_days
0,Home,47961,888.0,47.6,26.1,2.69
1,Google Online Store,31763,588.0,65.1,11.3,0.73
2,Apparel | Google Merchandise Store,15965,296.0,83.2,10.2,0.60
3,YouTube | Shop by Brand | Google Merchandise S...,13613,252.0,72.5,20.4,0.31
4,Google Dino Game Tee,9738,180.0,97.2,1.6,0.01
5,Men's T-Shirts | Apparel | Google Merchandise ...,3731,69.0,51.3,36.0,0.94


In [33]:
# Page addresses of top 2 landing pages
con.sql("""
    select s.landing_page_title, s.landing_page_location, count(*) as new_users
    from sessions s
    join new_users u
        on s.user_pseudo_id = u.user_pseudo_id
        and s.session_start = u.first_visit_start
    where s.landing_page_title in ('Home', 'Google Online Store')
    group by 1, 2
    order by 3 desc
    limit 4
""").df()


,landing_page_title,landing_page_location,new_users
0,Home,https://shop.googlemerchandisestore.com/,41960
1,Google Online Store,https://googlemerchandisestore.com/,19068
2,Google Online Store,https://www.googlemerchandisestore.com/,12682
3,Home,https://shop.googlemerchandisestore.com/store....,5967


In [34]:
# Check the stabilities of the arrivals and product viewed
con.sql("""
    select date_trunc('week', first_visit_start)::date as week_start,
        round(count(*) filter (where landing_page_title = 'Google Online Store') / 7.0) as entrance_per_day,
        round(100 * avg(viewed_product) filter (where landing_page_title = 'Google Online Store'), 1) as entrance_pct_product,
        round(count(*) filter (where landing_page_title = 'Google Dino Game Tee') / 7.0) as dino_per_day,
        round(100 * avg(viewed_product) filter (where landing_page_title = 'Google Dino Game Tee'), 1) as dino_pct_product
    from new_users
    where first_visit_start >= timestamp '2020-11-02'
        and first_visit_start < timestamp '2020-12-21'
    group by 1
    order by 1
""").df()

,week_start,entrance_per_day,entrance_pct_product,dino_per_day,dino_pct_product
0,2020-11-02,603.0,9.3,18.0,43.1
1,2020-11-09,567.0,8.8,26.0,3.8
2,2020-11-16,615.0,10.9,34.0,3.8
3,2020-11-23,650.0,11.2,30.0,1.9
4,2020-11-30,586.0,9.6,152.0,1.2
5,2020-12-07,572.0,10.0,466.0,0.6
6,2020-12-14,595.0,14.8,482.0,0.9


In [35]:
# Functions to define size
def users_per_group(rate, lift, alpha=0.05, power=0.80):
    """Users needed in each of two equal groups to detect a relative lift in a rate.
    Two-sided test of two proportions, normal approximation."""
    new_rate = rate * (1 + lift)
    z = NormalDist().inv_cdf(1 - alpha / 2) + NormalDist().inv_cdf(power)
    variance = rate * (1 - rate) + new_rate * (1 - new_rate)
    return math.ceil(z**2 * variance / (new_rate - rate) ** 2)


def days_needed(rate, lift, users_per_day):
    """Days of traffic to fill both groups."""
    return math.ceil(2 * users_per_group(rate, lift) / users_per_day)

In [ ]:
# For a 20% lift, how long time will a test need
print(
    "Entrance page, viewed a product (11.3%):",
    users_per_group(0.113, 0.20),
    "users per group,",
    days_needed(0.113, 0.20, 588),
    "days",
)
print("\nSame test if the rate is 9%:", days_needed(0.09, 0.20, 588), "days")
print(
    "\nEntrance page, purchased within 7 days (0.73%):",
    days_needed(0.0073, 0.20, 588),
    "days",
)
print("\nHome, viewed a product (26.1%):", days_needed(0.261, 0.20, 888), "days")

Entrance page, viewed a product (11.3%): 3342 users per group, 12 days

Same test if the rate is 9%: 15 days

Entrance page, purchased within 7 days (0.73%): 200 days

Home, viewed a product (26.1%): 3 days


Findings:
1. New users who land on Home viewed a product accounts for 26.1% against 11.3% on that landing on Google Online store.
    - Home: shop.googlemerchandisestore.com
    - Entrance page: googlemerchandisestore.com
2. 'Google Dino Game Tee' page increased sharply from 30 per day in November to 480 per day in mid of December and this is a point to check for more information before a test.
3. 65.1% users who land on the entrance page viewed one page or less so this is where visitors are lost. Entrance page traffic is stable between 567 to 650 per day. 
4. Time for a test to take: for entrance page arrivals, a 20% lift requires 12 days of traffic to measure viewing a product and 200 days for purchases.

Test settings:
- Target people: new users starting from the entrance page
- Randomly splitting into two groups 50% vs 50%:
    - A: people see the entrance page as today
    - B: people are sent to the homepage of the store
- Measurement: how many viewed a product in the first visit
- lift: smallest lift relative 20% worth detecting (11.3% to 13.6%).
- sample size: 3342 per group (5% significane, 80% power, two sided)
- test duration: two full weeks, fixed before start
- purchase is needed to be monitored as well, and at least not get worse.




Limitations:
- Here test is to follow product views not sales
- The gap between Home and entrance page cannot be seen as the effect of the change rather than the different visitors land on each page. A test is needed.

Next: Home as the largest landing page with 888 a day, a test needs 3 days of traffic within one full week.